In [ ]:
import numpy as np
import numpy.typing as npt
from shapely.wkt import loads
from shapely import Polygon
from bo_sheet.sim_wrapper_BO import _fric_var_1std_for_sim_wrapper, run_crawling_sim, _create_simulation_movie, render_clean_plate_png
import os
from typing import List
from bo_sheet.curve_compute_utils import round_down
import bo_sheet.channel_preprocess_utility as cu

material_dict = {}
fixed_parameters = {}
BO_parameters = {}

# [density (g/mm^3), Young's modulus (MPa), magnetization density (A/m)]
material_dict['13_9_2025_pdms_1_1'] = [1.795 * 1e-3, 2.2, 187298.43770657358]

BO_parameters['robot_length'] = 10.0
BO_parameters['robot_thickness'] = 0.1
BO_parameters['robot_n_waveform'] = 1.18
BO_parameters['robot_material'] = '13_9_2025_pdms_1_1'

fixed_parameters['robot_width'] = 2.0 # [mm]

fixed_parameters['target_dl'] = 0.25 # [mm]
fixed_parameters['cfl'] = 0.4

fixed_parameters['B_field'] = 0.05 # [T]
fixed_parameters['B_frequency'] = 4.0 # [Hz]

fixed_parameters['backward'] = False

# Here we want the robot to reach the end point of the channel
fixed_parameters['final_time'] = 30e3  # Must larger than 3000.0 [ms]

fixed_parameters['save_directory'] = os.path.join(os.path.expanduser('~'), 'Documents', 'GitHub', 'Bayesian_Optimization_for_Sheet_Robots', \
                              'code', '3_optim_shape')

fixed_parameters['segment_str_list'] = ['max_curvature', 'max_width', 'min_width', 'max_curvature_change', 'max_width_change']

fixed_parameters['variation_level'] = 0.5

# Varying friction coefficients at every contact step around the optimal friction coefficients
fixed_parameters['dynamic_fric_coeff'] = True
fixed_parameters['dynamic_fric_variation_level'] = 0.2

# Number of channels to be repeatedly constructed
fixed_parameters['n_channels'] = 1

# Number of varied friction cases to be done for a channel
fixed_parameters['n_trials'] = 1

fixed_parameters['upper_bound_robot_length'] = 10.0

fixed_parameters['static_mu_wall'] = 8.937293906750346
fixed_parameters['kinetic_mu_wall'] = 3.7862437848024757
fixed_parameters['static_mu_wall_end'] = 13.249480260330722
fixed_parameters['kinetic_mu_wall_end'] = 1.7875154798884043
fixed_parameters['static_mu_substrate'] = 0.12889650832825794
fixed_parameters['kinetic_mu_substrate'] = 0.017422378787529738
fixed_parameters['k_mag_den'] = 1.538629490914029
fixed_parameters['k_modulus'] = 1.1194380619912316

fixed_parameters['robot_density'], fixed_parameters['robot_modulus'], fixed_parameters['robot_magnetization_density'] = material_dict[BO_parameters['robot_material']]
fixed_parameters['end_line_percent'] = 1.0
fixed_parameters['offset_factor'] = 0.0 # Percentage of the segment midline length for the robot's tip closest to the start line position
fixed_parameters['precise_offset'] = 0.2 # [mm]

channel_polygon_list = []
x_midline_list = []
y_midline_list = []
left_bank_list = []
right_bank_list = []
total_length_list = []

for i in range(len(fixed_parameters['segment_str_list'])):
    with open(os.path.join(fixed_parameters['save_directory'], 'channel_params', 'coronary_artery',\
                           'zeroFour', 'trial_0',\
                            f"zeroFour_right_{fixed_parameters['segment_str_list'][i]}_polygon.wkt"), 'r') as file:
        channel_polygon = loads(file.read())
        file.close()

    with np.load(os.path.join(fixed_parameters['save_directory'], 'channel_params', 'coronary_artery',\
                           'zeroFour', 'trial_0',\
                            f"zeroFour_right_{fixed_parameters['segment_str_list'][i]}_channel_info.npz")) as channel_info:
        x_midline = channel_info['x_midline']
        y_midline = channel_info['y_midline']
        left_bank = channel_info['left_bank']
        right_bank = channel_info['right_bank']
        total_length = channel_info['total_length']
        channel_info.close()

    channel_polygon_list.append(channel_polygon)
    x_midline_list.append(x_midline)
    y_midline_list.append(y_midline)
    left_bank_list.append(left_bank)
    right_bank_list.append(right_bank)
    total_length_list.append(total_length)

fixed_parameters['channel_polygon_list'] = channel_polygon_list
fixed_parameters['x_midline_list'] = x_midline_list
fixed_parameters['y_midline_list'] = y_midline_list
fixed_parameters['left_bank_list'] = left_bank_list
fixed_parameters['right_bank_list'] = right_bank_list
fixed_parameters['total_length_list'] = total_length_list

robot_length: float = BO_parameters['robot_length']
robot_thickness: float = BO_parameters['robot_thickness']
robot_n_waveform: float = BO_parameters['robot_n_waveform']
robot_material: str = BO_parameters['robot_material']

robot_width: float = fixed_parameters['robot_width']
robot_density: float = fixed_parameters['robot_density']
robot_modulus: float = fixed_parameters['robot_modulus']
robot_magnetization_density: float = fixed_parameters['robot_magnetization_density']
target_dl: float = fixed_parameters['target_dl']
cfl: float = fixed_parameters['cfl']
B_field: float = fixed_parameters['B_field']
B_frequency: float = fixed_parameters['B_frequency']
backward: bool = fixed_parameters['backward']
final_time: float = fixed_parameters['final_time']
save_directory: str = fixed_parameters['save_directory']
variation_level: float = fixed_parameters['variation_level']
dynamic_fric_coeff: bool = fixed_parameters['dynamic_fric_coeff']
dynamic_fric_variation_level: float = fixed_parameters['dynamic_fric_variation_level']
n_channels: int = fixed_parameters['n_channels']
n_trials: int = fixed_parameters['n_trials']
upper_bound_robot_length: float = fixed_parameters['upper_bound_robot_length']
end_line_percent: float = fixed_parameters['end_line_percent']
offset_factor: float = fixed_parameters['offset_factor']
precise_offset: float = fixed_parameters['precise_offset']

channel_polygon_list: List[Polygon] = fixed_parameters['channel_polygon_list']
x_midline_list: List[npt.NDArray[np.float64]] = fixed_parameters['x_midline_list']
y_midline_list: List[npt.NDArray[np.float64]] = fixed_parameters['y_midline_list']
left_bank_list: List[npt.NDArray[np.float64]] = fixed_parameters['left_bank_list']
right_bank_list: List[npt.NDArray[np.float64]] = fixed_parameters['right_bank_list']
total_length_list: List[float] = fixed_parameters['total_length_list']

static_mu_wall: float = fixed_parameters['static_mu_wall']
kinetic_mu_wall: float = fixed_parameters['kinetic_mu_wall']
static_mu_wall_end: float = fixed_parameters['static_mu_wall_end']
kinetic_mu_wall_end: float = fixed_parameters['kinetic_mu_wall_end']
static_mu_substrate: float = fixed_parameters['static_mu_substrate']
kinetic_mu_substrate: float = fixed_parameters['kinetic_mu_substrate']
k_mag_den: float = fixed_parameters['k_mag_den']
k_modulus: float = fixed_parameters['k_modulus']

segment_str_list: List[str] = fixed_parameters['segment_str_list']

robot_magnetization_density *= k_mag_den
robot_modulus *= k_modulus

n_elem = int(robot_length / target_dl)
if n_elem * target_dl < robot_length:
    n_elem += 1
else:
    n_elem = n_elem

dl = robot_length / n_elem
dt = round_down(np.float64(cfl * dl / np.sqrt(robot_modulus / robot_density)))

dist_threshold = 0.01

for i in range(len(channel_polygon_list)):
    channel_seg_midline = np.column_stack((x_midline_list[i], y_midline_list[i]))

    robot_init_pos = cu.generate_fiber_in_segment(channel_seg_midline,
                                                L_fiber=robot_length,
                                                offset_factor=offset_factor,
                                                N_fiber=n_elem,
                                                precise_offset=precise_offset)
    
    robot_init_director = cu.compute_directors_from_positions(robot_init_pos)
    robot_init_origin = robot_init_pos[:, 0]

    friction_coefficient_arrays_list = _fric_var_1std_for_sim_wrapper(
        n_elem,
        static_mu_substrate,
        kinetic_mu_substrate,
        static_mu_wall,
        kinetic_mu_wall,
        static_mu_wall_end,
        kinetic_mu_wall_end,
        variation_level,
        n_trials
    )

    static_mu_substrate_array = friction_coefficient_arrays_list[0]['static_mu_substrate_array']
    kinetic_mu_substrate_array = friction_coefficient_arrays_list[0]['kinetic_mu_substrate_array']
    static_mu_wall_array = friction_coefficient_arrays_list[0]['static_mu_wall_array']
    kinetic_mu_wall_array = friction_coefficient_arrays_list[0]['kinetic_mu_wall_array']

    n_elem = robot_init_pos.shape[1] - 1
    magnetization_direction = np.zeros((3, n_elem))

    magnetization_angles = np.linspace(0, 2 * np.pi * robot_n_waveform, n_elem)

    magnetization_direction[0, :] = np.cos(magnetization_angles)
    magnetization_direction[1, :] = np.sin(magnetization_angles)

    post_processing_dict, actual_final_time_in_ms, reached_end_line = run_crawling_sim(
        length=robot_length,
        beam_width=robot_width,
        beam_thickness=robot_thickness,
        n_waveform=robot_n_waveform,
        beam_density=robot_density,
        modulus=robot_modulus,
        magnetization_density=robot_magnetization_density,
        channel_polygon=channel_polygon_list[i],
        dist_threshold=dist_threshold,
        robot_init_pos=robot_init_pos,
        rod_origin=robot_init_origin,
        rod_director=robot_init_director,
        B_field=B_field,
        B_frequency=B_frequency,
        kinetic_mu_substrate_array=kinetic_mu_substrate_array,
        static_mu_substrate_array=static_mu_substrate_array,
        kinetic_mu_wall_array=kinetic_mu_wall_array,
        static_mu_wall_array=static_mu_wall_array,
        dt=dt,
        final_time=final_time,
        magnetization_direction=magnetization_direction,
        variation_level=variation_level,
        dynamic_fric_coeff=dynamic_fric_coeff,
        dynamic_fric_variation_level=dynamic_fric_variation_level,
        backward=backward,
        move_end_line_to=end_line_percent,
        n_elem=n_elem
    )

    positions_over_time = np.array(post_processing_dict["position"])
    # np.savez(os.path.join(save_directory, 'channel_results', 'sinusoidal', 'trial_1', 'positions_over_time'), positions_over_time)
    # print(f"Travel time: {actual_final_time_in_ms * 1e-3}s")
    # print(f"Dealing with {segment_str_list[i]}")
    # render_clean_plate_png(channel_polygon_list[i], f"{segment_str_list[i]}", os.path.join(save_directory, 'channel_results', 'coronary_artery', 'zeroSix', 'trial_1'), include_title=False)

    _create_simulation_movie(positions_over_time, channel_polygon_list[i], f"{segment_str_list[i]}_sim_optimal_robot.mp4", os.path.join(save_directory, 'channel_results', 'coronary_artery', 'zeroFour', 'trial_0'), backward)

 17%|█▋        | 1912828/11538461 [05:02<25:20, 6330.49it/s]


End line (move_end_line_to 1.0) reached at time 4973.356! Minimum absolute distance away from the real end line: 0.498mm
Creating max_curvature_sim_optimal_robot.mp4 movie
Video saved to C:\Users\cwtseae\Documents\GitHub\Bayesian_Optimization_for_Sheet_Robots\code\3_optim_shape\channel_results\coronary_artery\zeroFour\trial_0\max_curvature_sim_optimal_robot.mp4


  8%|▊         | 939513/11538461 [02:25<27:17, 6472.38it/s]


End line (move_end_line_to 1.0) reached at time 2442.737! Minimum absolute distance away from the real end line: 0.500mm
Creating max_width_sim_optimal_robot.mp4 movie
Video saved to C:\Users\cwtseae\Documents\GitHub\Bayesian_Optimization_for_Sheet_Robots\code\3_optim_shape\channel_results\coronary_artery\zeroFour\trial_0\max_width_sim_optimal_robot.mp4


 26%|██▋       | 3044768/11538461 [07:42<21:31, 6577.53it/s]


End line (move_end_line_to 1.0) reached at time 7916.400! Minimum absolute distance away from the real end line: 0.500mm
Creating min_width_sim_optimal_robot.mp4 movie
Video saved to C:\Users\cwtseae\Documents\GitHub\Bayesian_Optimization_for_Sheet_Robots\code\3_optim_shape\channel_results\coronary_artery\zeroFour\trial_0\min_width_sim_optimal_robot.mp4


 23%|██▎       | 2625034/11538461 [06:33<22:14, 6677.25it/s]


End line (move_end_line_to 1.0) reached at time 6825.091! Minimum absolute distance away from the real end line: 0.500mm
Creating max_curvature_change_sim_optimal_robot.mp4 movie
Video saved to C:\Users\cwtseae\Documents\GitHub\Bayesian_Optimization_for_Sheet_Robots\code\3_optim_shape\channel_results\coronary_artery\zeroFour\trial_0\max_curvature_change_sim_optimal_robot.mp4


 25%|██▍       | 2827665/11538461 [06:57<21:26, 6769.66it/s]


End line (move_end_line_to 1.0) reached at time 7351.932! Minimum absolute distance away from the real end line: 0.499mm
Creating max_width_change_sim_optimal_robot.mp4 movie
Video saved to C:\Users\cwtseae\Documents\GitHub\Bayesian_Optimization_for_Sheet_Robots\code\3_optim_shape\channel_results\coronary_artery\zeroFour\trial_0\max_width_change_sim_optimal_robot.mp4
